In [1]:
import warnings
import logging

# Suppress Hugging Face and other warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("datasets").setLevel(logging.ERROR)
logging.getLogger("evaluate").setLevel(logging.ERROR)



In [ ]:
from huggingface_hub import login

login("hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

hf_token = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"

In [7]:
#!pip install langchain-google-genai langchain faiss-cpu sentence-transformers groq dspy-ai


In [3]:
import os
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate
import requests

In [ ]:
# 1. Set keys
os.environ["GOOGLE_API_KEY"] = "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
os.environ["GROQ_API_KEY"] = "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"

In [15]:
import pandas as pd

# 2. Load data (replace these with your files)
df_passages = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
df_test = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

In [17]:
df_test.head(10)

,question,answer,relevant_passage_ids
id,,,
0,Is Hirschsprung disease a mendelian or a multi...,"Coding sequence mutations in RET, GDNF, EDNRB,...","[20598273, 6650562, 15829955, 15617541, 230011..."
1,List signaling molecules (ligands) that intera...,The 7 known EGFR ligands are: epidermal growt...,"[23821377, 24323361, 23382875, 22247333, 23787..."
2,Is the protein Papilin secreted?,"Yes, papilin is a secreted protein","[21784067, 19297413, 15094122, 7515725, 332004..."
3,Are long non coding RNAs spliced?,Long non coding RNAs appear to be spliced thro...,"[22955974, 21622663, 22707570, 22955988, 24285..."
4,Is RANKL secreted from the cells?,Receptor activator of nuclear factor κB ligand...,"[22867712, 23827649, 21618594, 23835909, 24265..."
5,Does metformin interfere thyroxine absorption?,No. There are not reported data indicating tha...,[26191653]
6,Which miRNAs could be used as potential biomar...,"miR-200a, miR-100, miR-141, miR-200b, miR-200c...","[23918241, 23621186, 22246341, 23978303, 23888..."
7,Which acetylcholinesterase inhibitors are used...,Pyridostigmine and neostygmine are acetylcholi...,"[21328290, 21133188, 15610702, 20663605, 21815..."
8,Has Denosumab (Prolia) been approved by FDA?,"Yes, Denosumab was approved by the FDA in 2010.","[24114694, 22540167, 21129866, 21170699, 23956..."


In [19]:
df_test.shape

(4719, 3)

In [21]:
df_passages = df_passages.reset_index()
df_passages = df_passages.reset_index()

df_passages = df_passages.rename(columns={'id': 'id'})
df_passages = df_passages.rename(columns={'id': 'id'})


In [23]:
df_passages.head(10)

,index,id,passage
0,0,9797,New data on viruses isolated from patients wit...
1,1,11906,We describe an improved method for detecting d...
2,2,16083,We have studied the effects of curare on respo...
3,3,23188,Kinetic and electrophoretic properties of 230-...
4,4,23469,Male Wistar specific-pathogen-free rats aged 2...
5,5,24032,Tyrosine hydroxylase (TH) and phenylethanolami...
6,6,30666,Hemolytic anemia is a well-recognized complica...
7,7,58611,(1) The RNA replicase induced by bacteriophage...
8,8,61441,Mice were inoculated with human sarcoid tissue...
9,9,83311,Bleomycin is potentially capable of inducing a...


In [25]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from tqdm import tqdm

nltk.download('punkt')
# Token-based chunking function
def chunk_text_token_limit(text, max_tokens=200):
    tokens = word_tokenize(text)
    chunks = []
    for i in range(0, len(tokens), max_tokens):
        chunk = tokens[i:i+max_tokens]
        chunks.append(" ".join(chunk))
    return chunks

# Apply chunking to 'text' column
tqdm.pandas(desc="Chunking Passages")
df_passages['token_chunks'] = df_passages['passage'].astype(str).progress_apply(lambda x: chunk_text_token_limit(x, max_tokens=200))

# Flatten chunks for embedding
all_token_chunks = [chunk for chunk_list in df_passages['token_chunks'] for chunk in chunk_list]


[nltk_data] Downloading package punkt to /Users/mukulgarg/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Chunking Passages: 100%|████████████████| 40221/40221 [00:10<00:00, 3845.27it/s]


In [26]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Generating embeddings...")
embeddings = embedding_model.embed_documents(all_token_chunks)


Generating embeddings...


In [28]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
from tqdm import tqdm

# Setup
nltk.download('punkt')
# --- Step 1: Token-based Chunking (max 200 tokens) ---

def chunk_text_token_limit(text, passage_id, max_tokens=200):
    tokens = word_tokenize(text)
    chunks = []
    for i in range(0, len(tokens), max_tokens):
        chunk_tokens = tokens[i:i+max_tokens]
        chunk_text = " ".join(chunk_tokens)
        chunks.append((f"{passage_id}_{i//max_tokens}", chunk_text))  # Unique chunk ID
    return chunks

# Apply chunking to all passages
all_chunks = []
for _, row in tqdm(df_passages.iterrows(), total=len(df_passages), desc="Chunking passages"):
    passage_id = row['id']
    text = str(row['passage'])  # Ensure it's a string
    chunks = chunk_text_token_limit(text, passage_id)
    all_chunks.extend(chunks)

chunk_ids = [chunk[0] for chunk in all_chunks]
chunk_texts = [chunk[1] for chunk in all_chunks]

# --- Step 2: Embedding with SentenceTransformer ---

print("Encoding chunked passages...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = embedder.encode(chunk_texts, show_progress_bar=True)
chunk_embeddings = np.array(chunk_embeddings).astype('float32')

# --- Step 3: Create FAISS Index with IDs ---

dimension = chunk_embeddings.shape[1]
index_flat = faiss.IndexFlatL2(dimension)
index = faiss.IndexIDMap(index_flat)

# Convert string IDs to integers (optional: keep mapping in a dict)
chunk_id_map = {i: chunk_ids[i] for i in range(len(chunk_ids))}
int_ids = np.array(list(chunk_id_map.keys())).astype('int64')

# Add to index
index.add_with_ids(chunk_embeddings, int_ids)

print(f"FAISS index created with {len(chunk_ids)} chunked passages.")


[nltk_data] Downloading package punkt to /Users/mukulgarg/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Chunking passages: 100%|████████████████| 40221/40221 [00:12<00:00, 3338.91it/s]


Encoding chunked passages...


Batches:   0%|          | 0/1908 [00:00<?, ?it/s]

FAISS index created with 61047 chunked passages.


In [29]:
# embedder = SentenceTransformer('all-MiniLM-L6-v2')
# passages = df_passages['passage'].tolist()
# ids = df_passages['id'].tolist()
# passage_embeds = embedder.encode(passages, show_progress_bar=True)
# dimension = passage_embeds.shape[1]
# faiss_index = faiss.IndexFlatL2(dimension)
# faiss_index.add(np.array(passage_embeds))
# print("Faiss Index Created")

In [116]:
def retrieve_top_k(query, k=2):
    # Encode query
    query_embed = embedder.encode([query]).astype('float32')
    
    # Search index
    D, I = index.search(query_embed, k)
    results = []
    for idx, score in zip(I[0], D[0]):
        if idx < len(chunk_texts):
            result = {
                "chunk_id": chunk_id_map.get(idx, str(idx)),
                "chunk_text": chunk_texts[idx],
                "score": float(score)
            }
            results.append(result)
    return results

In [31]:
# # Query rewriting (Gemini/Gemma via LangChain)
# llm_rewriter = ChatGoogleGenerativeAI(model="gemini-1.5-flash-latest")  # Or "gemma-7b-it"
# rewrite_prompt = PromptTemplate.from_template(
#     """Provide several specific rewritten versions of the biomedical question, ranging from broad to precise. Output as 'Option 1', 'Option 2', etc.
# Question: {question}
# Rewritten:"""
# )
# rewrite_chain = rewrite_prompt | llm_rewriter

In [32]:
# def rewrite_query(question, preferred_option="Option 2"):
#     output = rewrite_chain.invoke({"question": question}).content
#     for line in output.split("\n"):
#         if line.strip().startswith(preferred_option):
#             return line.split(":", 1)[1].strip()
#     for line in output.split("\n"):
#         if "Option" in line and ":" in line:
#             return line.split(":", 1)[1].strip()
#     return output.strip()

In [33]:
# def answer_with_gemini(question, context):
#     guarded_prompt = (
#         "You are a biomedical expert. Answer the following question using ONLY the context below. "
#         "If the answer is not present in the context, reply: \"I'm sorry, I cannot answer that question based on the provided information.\"\n"
#         f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
#     )
#     return llm.invoke(guarded_prompt).content.strip()

In [34]:
import requests

GROQ_API_KEY = os.environ['GROQ_API_KEY']
  # Get at https://console.groq.com/

import requests

def answer_with_groq(question, context):
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }
    prompt = (
        "You are a biomedical expert. Answer the following question using ONLY the context below. "
        "If the answer is not present in the context, reply: \"I'm sorry, I cannot answer that question based on the provided information.\"\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    data = {
        "model": "llama3-70b-8192",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 256
    }
    resp = requests.post(url, headers=headers, json=data)
    try:
        out = resp.json()
        if 'choices' in out:
            return out['choices'][0]['message']['content'].strip(), None
        elif 'error' in out:
            # Try to extract suggested wait time
            wait_seconds = 10  # Default if not found
            if 'message' in out['error']:
                import re
                match = re.search(r'try again in ([\d\.]+)s', out['error']['message'])
                if match:
                    wait_seconds = float(match.group(1))
            #print("Groq API error:", out['error'])
            return API_ERROR, wait_seconds
        else:
            #print("Unexpected Groq API response:", out)
            return API_CALL_FAILED, 10
    except Exception as e:
        #print("Groq API response error:", e)
        #print(resp.text)
        return API_CALL_FAILED, 10



In [ ]:
def rag_pipeline(question, k=2, use_groq=True):
    # Rewrite query using Groq-based query rewriting function
    rewritten = rewrite_query_groq(question)
    
    # Retrieve top-k relevant chunks as list of dicts
    retrieved = retrieve_top_k(rewritten, k)
    
    # Join only the text content for context
    context = "\n".join([item["chunk_text"] for item in retrieved])
    
    if use_groq:
        # answer_with_groq returns (answer, wait_time)
        answer, wait_time = answer_with_groq(question, context)
        if wait_time is not None:
            # Could handle retry or log quota wait time here
            print(f"Groq API rate limited. Suggested wait: {wait_time}s")
    else:
        answer = answer_with_gemini(question, context)
    
    return {
        "question": question,
        "rewritten": rewritten,
        "context": context,
        "answer": answer
    }


## Example using Gemini

In [57]:
result = rag_pipeline("What are the symptoms of COVID-19?",use_groq=True)
print(ORIGINAL_QUESTION, result["question"])
print("\n\n")
print(REWRITTEN_QUESTION, result["rewritten"])
print("\n\n")
print(RETRIEVED_CONTEXT, result["context"])
print("\n\n")
print(ANSWER_LABEL, result["answer"])

Original Question: What are the symptoms of COVID-19?



Rewritten: What are the clinical manifestations or signs of SARS-CoV-2 infection in humans?

This rewritten question is more specific and uses technical terms commonly used in biomedical literature, making it more likely to retrieve relevant and accurate passages from a biomedical database or search engine.



Retrieved Context:
 BACKGROUND AND AIMS : Long COVID is the collective term to denote persistence of symptoms in those who have recovered from SARS-CoV-2 infection . METHODS : WE searched the pubmed and scopus databases for original articles and reviews . Based on the search result , in this review article we are analyzing various aspects of Long COVID . RESULTS : Fatigue , cough , chest tightness , breathlessness , palpitations , myalgia and difficulty to focus are symptoms reported in long COVID . It could be related to organ damage , post viral syndrome , post-critical care syndrome and others . Clinical evaluation shoul

## Example using Groq

In [59]:
result = rag_pipeline("Who is the president of US?",use_groq=True)
print(ORIGINAL_QUESTION, result["question"])
print("\n\n")
print(REWRITTEN_QUESTION, result["rewritten"])
print("\n\n")
print(RETRIEVED_CONTEXT, result["context"])
print("\n\n")
print(ANSWER_LABEL, result["answer"])

Original Question: Who is the president of US?



Rewritten: OUT OF SCOPE



Retrieved Context:
 development .
development .



Answer:
 I'm sorry, I cannot answer that question based on the provided information.


In [61]:
result = rag_pipeline("What are the symptoms of COVID-19?",use_groq=True)
print(ORIGINAL_QUESTION, result["question"])
print("\n\n")
print(REWRITTEN_QUESTION, result["rewritten"])
print("\n\n")
print(RETRIEVED_CONTEXT, result["context"])
print("\n\n")
print(ANSWER_LABEL, result["answer"])

Original Question: What are the symptoms of COVID-19?



Rewritten: What are the clinical manifestations of SARS-CoV-2 infection?



Retrieved Context:
 BACKGROUND AND AIMS : Long COVID is the collective term to denote persistence of symptoms in those who have recovered from SARS-CoV-2 infection . METHODS : WE searched the pubmed and scopus databases for original articles and reviews . Based on the search result , in this review article we are analyzing various aspects of Long COVID . RESULTS : Fatigue , cough , chest tightness , breathlessness , palpitations , myalgia and difficulty to focus are symptoms reported in long COVID . It could be related to organ damage , post viral syndrome , post-critical care syndrome and others . Clinical evaluation should focus on identifying the pathophysiology , followed by appropriate remedial measures . In people with symptoms suggestive of long COVID but without known history of previous SARS-CoV-2 infection , serology may help confirm the diagnos

# Evaluation 

In [64]:
# !pip install rouge-score evaluate
# !pip install bert-score

In [50]:
def rewrite_query_groq(question, return_wait=False):
    import os, requests, re
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {os.environ['GROQ_API_KEY']}",
        "Content-Type": "application/json"
    }
    prompt = (
        "Rewrite the following biomedical question for optimal passage retrieval. "
        "If the question is vague or not about biomedicine, reply: OUT OF SCOPE.\n"
        f"Question: {question}\nRewritten:"
    )
    data = {
        "model": "llama3-70b-8192",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 64
    }
    resp = requests.post(url, headers=headers, json=data)
    try:
        out = resp.json()
        if "choices" in out:
            rewritten = out["choices"][0]["message"]["content"].strip()
            return (rewritten, None) if return_wait else rewritten
        elif "error" in out:
            wait_seconds = 10
            if "message" in out["error"]:
                match = re.search(r'try again in ([\d\.]+)s', out["error"]["message"])
                if match:
                    wait_seconds = float(match.group(1))
            #print("Groq error:", out["error"])
            return (API_ERROR, wait_seconds) if return_wait else API_ERROR
        else:
            #print("Unexpected Groq API response:", out)
            return (API_CALL_FAILED, 10) if return_wait else API_CALL_FAILED
    except Exception as e:
        #print("Groq API response error:", e)
        #print(resp.text)
        return (API_CALL_FAILED, 10) if return_wait else API_CALL_FAILED


In [108]:
from nltk.tokenize import word_tokenize
import time
from collections import deque

MAX_REQUESTS_PER_MINUTE = 30
MAX_TOKENS_PER_MINUTE = 6000
MAX_OUTPUT_TOKENS = 256  # Adjust based on your max_tokens param in API

request_timestamps = deque()
token_usage = deque()

def count_tokens(text):
    tokens = word_tokenize(text)
    return len(tokens)

def can_send_request(input_tokens):
    now = time.time()
    # Remove entries older than 60 seconds
    while request_timestamps and now - request_timestamps[0] > 60:
        request_timestamps.popleft()
        token_usage.popleft()
    tokens_last_min = sum(token_usage)
    # Check both request count and token count limits
    if len(request_timestamps) < MAX_REQUESTS_PER_MINUTE and (tokens_last_min + input_tokens + MAX_OUTPUT_TOKENS) <= MAX_TOKENS_PER_MINUTE:
        return True
    return False

def wait_for_slot(input_tokens):
    while not can_send_request(input_tokens):
        time.sleep(0.5)

def record_request(input_tokens):
    request_timestamps.append(time.time())
    token_usage.append(input_tokens + MAX_OUTPUT_TOKENS)


# Throttled API call wrappers

def throttled_answer_with_groq(question, context):
    input_text = f"Question: {question}\nContext:\n{context}"
    input_tokens = count_tokens(input_text)

    wait_for_slot(input_tokens)
    answer, wait_time = answer_with_groq(question, context)
    record_request(input_tokens)

    return answer, wait_time


def throttled_rewrite_query_groq(question):
    input_tokens = count_tokens(question)  # or more precise prompt tokens if available

    wait_for_slot(input_tokens)
    rewritten, wait_time = rewrite_query_groq(question, return_wait=True)
    record_request(input_tokens)

    return rewritten, wait_time


In [127]:
import os
import pandas as pd
import time
from tqdm import tqdm

CHECKPOINT_FILE = "rag_predictions_checkpoint2.pkl"
SAVE_EVERY = 10  # Save every 10 questions

# Create mapping from passage text to its ID
text_to_id = dict(zip(df_passages["passage"], df_passages["id"]))

# Try to load previous progress
if os.path.exists(CHECKPOINT_FILE):
    checkpoint = pd.read_pickle(CHECKPOINT_FILE)
    processed_idxs = set(checkpoint['idx'].tolist())
    pred_answers_baseline = checkpoint['baseline'].tolist()
    pred_answers_rag = checkpoint['rag'].tolist()
    pred_passage_ids_baseline = checkpoint['baseline_pids'].tolist()
    pred_passage_ids_rag = checkpoint['rag_pids'].tolist()
    print(f"Loaded {len(processed_idxs)} previously processed entries")
else:
    processed_idxs = set()
    pred_answers_baseline = []
    pred_answers_rag = []
    pred_passage_ids_baseline = []
    pred_passage_ids_rag = []

for idx, row in tqdm(df_test.iterrows(), total=len(df_test)):
    if idx in processed_idxs:
        continue

    question = row['question']

    # --- BASELINE ---
    retrieved_passages_base = retrieve_top_k(question, k=3)
    context_baseline = "\n".join([p["chunk_text"] for p in retrieved_passages_base])

    while True:
        answer_base, wait_time = throttled_answer_with_groq(question, context_baseline)
        if answer_base in (API_ERROR, API_CALL_FAILED):
            print(f"[Baseline] API error, retrying after {wait_time}s for question idx={idx}")
            time.sleep(wait_time)
            continue
        break


    retrieved_ids_base = [p["chunk_id"] for p in retrieved_passages_base]

    # --- RAG ---
    while True:
        rewritten, wait_time = throttled_rewrite_query_groq(question)
        if rewritten in (API_ERROR, API_CALL_FAILED):
            time.sleep(wait_time)
            continue
        break

    retrieved_passages_rag = retrieve_top_k(rewritten, k=3)
    context_rag = "\n".join([p["chunk_text"] for p in retrieved_passages_rag])

    while True:
        answer_rag, wait_time = throttled_answer_with_groq(question, context_rag)
        if answer_rag in (API_ERROR, API_CALL_FAILED):
            time.sleep(wait_time)
            continue
        break

    retrieved_ids_rag = [p["chunk_id"] for p in retrieved_passages_rag]

    # Save results in memory
    pred_answers_baseline.append(answer_base)
    pred_answers_rag.append(answer_rag)
    pred_passage_ids_baseline.append(retrieved_ids_base)
    pred_passage_ids_rag.append(retrieved_ids_rag)
    processed_idxs.add(idx)

    # Save checkpoint every SAVE_EVERY questions
    if idx % SAVE_EVERY == 0:
        df_checkpoint = pd.DataFrame({
            "idx": list(processed_idxs),
            "baseline": pred_answers_baseline,
            "rag": pred_answers_rag,
            "baseline_pids": pred_passage_ids_baseline,
            "rag_pids": pred_passage_ids_rag,
        })
        df_checkpoint.to_pickle(CHECKPOINT_FILE)

# Final save
df_checkpoint = pd.DataFrame({
    "idx": list(processed_idxs),
    "baseline": pred_answers_baseline,
    "rag": pred_answers_rag,
    "baseline_pids": pred_passage_ids_baseline,
    "rag_pids": pred_passage_ids_rag,
})
df_checkpoint.to_pickle(CHECKPOINT_FILE)


Loaded 1 previously processed entries


  0%|                                                  | 0/4719 [00:00<?, ?it/s]

[Baseline] API error, retrying after 10s for question idx=1
[Baseline] API error, retrying after 10s for question idx=1
[Baseline] API error, retrying after 58.8142s for question idx=1


  0%|                                     | 1/4719 [01:42<134:14:01, 102.43s/it]


KeyboardInterrupt: 

In [137]:
import os
import pickle
from tqdm import tqdm

checkpoint_file = "rag_evaluation_checkpoint.pkl"

# Load checkpoint if exists
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, "rb") as f:
        checkpoint = pickle.load(f)
        pred_answers_baseline = checkpoint.get("pred_answers_baseline", [])
        pred_answers_rag = checkpoint.get("pred_answers_rag", [])
        pred_passage_ids_baseline = checkpoint.get("pred_passage_ids_baseline", [])
        pred_passage_ids_rag = checkpoint.get("pred_passage_ids_rag", [])
        start_idx = len(pred_answers_baseline)
        print(f"Loaded checkpoint. Resuming from index {start_idx}.")
else:
    pred_answers_baseline = []
    pred_answers_rag = []
    pred_passage_ids_baseline = []
    pred_passage_ids_rag = []
    start_idx = 0
    print("No checkpoint found. Starting from scratch.")

TOP_K = 2

for idx, row in tqdm(list(df_test.iterrows())[start_idx:], total=len(df_test) - start_idx):
    question = row['question']

    try:
        # Baseline pipeline
        retrieved_base = retrieve_top_k(question, k=TOP_K)
        context_base = "\n".join([p["chunk_text"] for p in retrieved_base])
        answer_base = throttled_answer_with_groq(question, context_base)[0]
        ids_base = [p["chunk_id"] for p in retrieved_base]

        # RAG pipeline
        rewritten = throttled_rewrite_query_groq(question)[0]
        retrieved_rag = retrieve_top_k(rewritten, k=TOP_K)
        context_rag = "\n".join([p["chunk_text"] for p in retrieved_rag])
        answer_rag = throttled_answer_with_groq(question, context_rag)[0]
        ids_rag = [p["chunk_id"] for p in retrieved_rag]

        # Save results
        pred_answers_baseline.append(answer_base)
        pred_answers_rag.append(answer_rag)
        pred_passage_ids_baseline.append(ids_base)
        pred_passage_ids_rag.append(ids_rag)

    except Exception as e:
        print(f"[ERROR] Failed at index {start_idx + idx}: {e}")
        break  # or continue if you want to skip errors

    # Save checkpoint every 10 entries
    if (len(pred_answers_baseline) % 10) == 0:
        with open(checkpoint_file, "wb") as f:
            pickle.dump({
                "pred_answers_baseline": pred_answers_baseline,
                "pred_answers_rag": pred_answers_rag,
                "pred_passage_ids_baseline": pred_passage_ids_baseline,
                "pred_passage_ids_rag": pred_passage_ids_rag
            }, f)
        print(f"[Checkpoint Saved] Up to index {start_idx + idx}")


Loaded checkpoint. Resuming from index 10.


  0%|                                      | 10/4709 [02:52<23:36:22, 18.09s/it]

[Checkpoint Saved] Up to index 29


  0%|▏                                     | 20/4709 [04:54<15:11:03, 11.66s/it]

[Checkpoint Saved] Up to index 39


...

[Checkpoint Saved] Up to index 49


  1%|▎                                     | 40/4709 [08:58<10:42:29,  8.26s/it]

[Checkpoint Saved] Up to index 59


  1%|▎                                     | 46/4709 [10:06<17:04:01, 13.18s/it]


KeyboardInterrupt: 

In [139]:
# Number of evaluated queries
num_evaluated = len(pred_answers_baseline)

# Get the corresponding rows from df_test
df_evaluated = df_test.iloc[:num_evaluated].copy()

# Extract gold answers
gold_answers = df_evaluated['answer'].tolist()
import pandas as pd

df_results = pd.DataFrame({
    "question": df_evaluated['question'].tolist(),
    "gold_answer": gold_answers,
    "baseline_answer": pred_answers_baseline,
    "rag_answer": pred_answers_rag,
    "baseline_passage_ids": pred_passage_ids_baseline,
    "rag_passage_ids": pred_passage_ids_rag
})


In [ ]:

from bert_score import score as bert_score
from sklearn.metrics import average_precision_score
import numpy as np

# 1. ROUGE Score
import evaluate
rouge = evaluate.load("rouge", verbose=False)
bertscore = evaluate.load("bertscore", verbose=False)

rouge_baseline = rouge.compute(predictions=pred_answers_baseline, references=gold_answers, use_stemmer=True)
rouge_rag = rouge.compute(predictions=pred_answers_rag, references=gold_answers, use_stemmer=True)

print("\n📊 ROUGE-L F1:")

print(f"Baseline: {rouge_baseline['rougeL']:.4f}")
print(f"RAG     : {rouge_rag['rougeL']:.4f}")

# 2. BERTScore

P_base, R_base, F1_base = bert_score(pred_answers_baseline, gold_answers, lang="en", verbose=False)
P_rag, R_rag, F1_rag = bert_score(pred_answers_rag, gold_answers, lang="en", verbose=False)

print("\n📊 BERTScore F1 (avg):")
print(f"BERTScore F1: {sum(bert_base['f1']) / len(bert_base['f1']):.4f}")
print(f"Baseline: {F1_base['f1'].mean():.4f}")
print(f"RAG     : {F1_rag['f1'].mean():.4f}")


def compute_map_mrr(pred_ids_list, df_eval, k=3):
    """pred_ids_list: list of lists of predicted passage IDs
       df_eval: dataframe with column `relevant_passage_ids`"""
    map_scores = []
    mrr_scores = []

    for i, pred_ids in enumerate(pred_ids_list):
        relevant = set(df_eval.iloc[i]["relevant_passage_ids"])

        if not relevant:
            continue  # skip if no gold passages

        hits = [1 if pid in relevant else 0 for pid in pred_ids]

        # MAP@k
        precisions = [
            sum(hits[:i + 1]) / (i + 1) for i in range(len(hits)) if hits[i]
        ]
        ap = np.mean(precisions) if precisions else 0.0
        map_scores.append(ap)

        # MRR@k
        rr = next((1 / (i + 1) for i, hit in enumerate(hits) if hit), 0.0)
        mrr_scores.append(rr)

    return np.mean(map_scores), np.mean(mrr_scores)

# Make sure df_evaluated is aligned
map_base, mrr_base = compute_map_mrr(pred_passage_ids_baseline, df_evaluated)
map_rag, mrr_rag = compute_map_mrr(pred_passage_ids_rag, df_evaluated)

print("\n📊 MAP / MRR (passage retrieval):")
print(f"Baseline MAP: {map_base:.4f}, MRR: {mrr_base:.4f}")
print(f"RAG      MAP: {map_rag:.4f}, MRR: {mrr_rag:.4f}")


In [349]:
import pandas as pd

# Load saved checkpoint
checkpoint_df = pd.read_pickle("rag_predictions_checkpoint.pkl")
print(checkpoint_df.columns)


Index(['idx', 'baseline', 'rag', 'baseline_pids', 'rag_pids'], dtype='object')


In [351]:
# Ensure gold answers are aligned with prediction indices
gold_answers = df_test.loc[checkpoint_df['idx'], 'answer'].reset_index(drop=True)

# Also align predicted outputs
baseline_answers = checkpoint_df['baseline'].reset_index(drop=True)
rag_answers = checkpoint_df['rag'].reset_index(drop=True)


In [389]:
import pandas as pd

# Load checkpoint and df_test (already available in your session)
checkpoint = pd.read_pickle("rag_predictions_checkpoint.pkl")

# Fetch gold/reference answers from df_test using stored idx
checkpoint["reference"] = checkpoint["idx"].apply(lambda i: df_test.loc[i, "answer"])

checkpoint.to_csv("routput.csv", index=False)
print("Saved as rag_evaluation_output.csv")

Saved as rag_evaluation_output.csv


In [353]:
# for gold, pred in zip(gold_answers, rag_answers):
#     print(f"Gold: {gold}\nRAG: {pred}\n---")


In [355]:
import evaluate
rouge = evaluate.load("rouge", verbose=False)
bertscore = evaluate.load("bertscore", verbose=False)


# Compute scores
rouge_result = rouge.compute(predictions=rag_answers, references=gold_answers, rouge_types=["rougeL"])
bert_result = bert.compute(predictions=rag_answers, references=gold_answers, lang="en")

print("ROUGE-L:", rouge_result["rougeL"])
print("BERT-F1:", sum(bert_result["f1"]) / len(bert_result["f1"]))


ROUGE-L: 0.19718255560488385
BERT-F1: 0.8505009488013892


In [357]:
import pandas as pd
import evaluate

# Load predictions
checkpoint = pd.read_pickle("rag_predictions_checkpoint.pkl")
df_eval = df_test.loc[checkpoint["idx"]].copy()
df_eval["baseline_pred"] = checkpoint["baseline"]
df_eval["rag_pred"] = checkpoint["rag"]

# Reference answers
references = df_eval["answer"].tolist()

# Metrics
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")


# --- Baseline Evaluation ---
print("\n--- Baseline ---")
rouge_base = rouge.compute(predictions=df_eval["baseline_pred"], references=references)
bert_base = bertscore.compute(predictions=df_eval["baseline_pred"], references=references, lang="en")


print(f"ROUGE-L: {rouge_base['rougeL']:.4f}")
print(f"BERTScore F1: {sum(bert_base['f1']) / len(bert_base['f1']):.4f}")


# --- RAG Evaluation ---
print("\n--- RAG ---")
rouge_rag = rouge.compute(predictions=df_eval["rag_pred"], references=references)
bert_rag = bertscore.compute(predictions=df_eval["rag_pred"], references=references, lang="en")


print(f"ROUGE-L: {rouge_rag['rougeL']:.4f}")
print(f"BERTScore F1: {sum(bert_rag['f1']) / len(bert_rag['f1']):.4f}")




--- Baseline ---
ROUGE-L: 0.2142
BERTScore F1: 0.8549

--- RAG ---
ROUGE-L: 0.1978
BERTScore F1: 0.8505


In [337]:
checkpoint.head(10)

,idx,baseline,rag,baseline_pids,rag_pids
0,0,"Based on the provided context, the answer is:\...","According to the context, Hirschsprung disease...","[22260, 435]","[22260, 21024]"
1,1,"Based on the provided context, the signaling m...","According to the provided context, the signali...","[23647, 32793]","[23647, 40093]"
2,2,"Yes, Papilins are secreted extracellular matri...","According to the provided context, the answer ...","[7017, 1072]","[7017, 4422]"
3,3,"I'm sorry, I cannot answer that question based...","I'm sorry, I cannot answer that question based...","[27685, 22090]","[27685, 26224]"
4,4,"I'm sorry, I cannot answer that question based...","I'm sorry, I cannot answer that question based...","[187, 17761]","[4180, 13398]"
5,5,"I'm sorry, I cannot answer that question based...","I'm sorry, I cannot answer that question based...","[10925, 6810]","[6810, 10925]"
6,6,"Based on the provided information, miRNAs-21, ...","Based on the provided context, the miRNAs that...","[25409, 12686]","[12686, 23239]"
7,7,"Based on the provided context, the answer is: ...","Based on the provided information, the answer ...","[36239, 16113]","[36239, 16113]"
8,8,"Yes, Denosumab (Prolia) has been approved by t...","Yes, Denosumab (Prolia) was approved by the FD...","[19296, 17063]","[19296, 17063]"
9,9,"I'm sorry, I cannot answer that question based...",The human genes encoding for the dishevelled p...,"[23671, 13416]","[2501, 23671]"


In [339]:
# Replace passage indices with actual passage IDs
index_to_id = dict(enumerate(df_passages["id"]))

# Replace indices in baseline_pids and rag_pids
resolved_baseline_pids = [[index_to_id[i] for i in ids] for ids in pred_passage_ids_baseline]
resolved_rag_pids = [[index_to_id[i] for i in ids] for ids in pred_passage_ids_rag]

# Save checkpoint with resolved IDs
pd.DataFrame({
    "idx": list(processed_idxs),
    "baseline": pred_answers_baseline,
    "rag": pred_answers_rag,
    "baseline_pids": resolved_baseline_pids,
    "rag_pids": resolved_rag_pids,
}).to_pickle(CHECKPOINT_FILE)

print("Final checkpoint saved with actual passage IDs!")


Final checkpoint saved with actual passage IDs!


In [347]:
# Load predictions
checkpoint = pd.read_pickle("rag_predictions_checkpoint.pkl")
checkpoint.head(5)

,idx,baseline,rag,baseline_pids,rag_pids
0,0,"Based on the provided context, the answer is:\...","According to the context, Hirschsprung disease...","[23001136, 1785632]","[23001136, 22584707]"
1,1,"Based on the provided context, the signaling m...","According to the provided context, the signali...","[23382875, 27426127]","[23382875, 34667080]"
2,2,"Yes, Papilins are secreted extracellular matri...","According to the provided context, the answer ...","[15094122, 3320045]","[15094122, 11076767]"
3,3,"I'm sorry, I cannot answer that question based...","I'm sorry, I cannot answer that question based...","[24655717, 22955974]","[24655717, 24130305]"
4,4,"I'm sorry, I cannot answer that question based...","I'm sorry, I cannot answer that question based...","[1334264, 21445329]","[10837071, 19302050]"


In [ ]:
import math

def compute_map_mrr(true_passage_ids, predicted_passage_ids):
    map_scores = []
    mrr_scores = []

    for true_ids, pred_ids in zip(true_passage_ids, predicted_passage_ids):
        ap = 0.0
        correct = 0
        rr = 0.0
        for i, pid in enumerate(pred_ids):
            if pid in true_ids:
                correct += 1
                ap += correct / (i + 1)
                if math.isclose(rr, 0.0):
                    rr = 1.0 / (i + 1)
        if correct > 0:
            ap /= correct
        map_scores.append(ap)
        mrr_scores.append(rr)

    return round(sum(map_scores) / len(map_scores), 4), round(sum(mrr_scores) / len(mrr_scores), 4)


In [361]:
checkpoint_df.sort_values("idx", inplace=True)

# Make sure `df_test` matches the order
df_test_sorted = df_test.reset_index().iloc[checkpoint_df['idx']].reset_index(drop=True)

# Add gold relevant_passage_ids to checkpoint_df
checkpoint_df["gold_passage_ids"] = df_test_sorted["relevant_passage_ids"]


In [363]:
import ast

# Safely convert string-represented lists into actual Python lists
checkpoint_df["gold_passage_ids"] = checkpoint_df["gold_passage_ids"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)


In [365]:
# Compute for baseline
map_base, mrr_base = compute_map_mrr(checkpoint_df["gold_passage_ids"], checkpoint_df["baseline_pids"])
print(f"Baseline → MAP: {map_base}, MRR: {mrr_base}")

# Compute for RAG
map_rag, mrr_rag = compute_map_mrr(checkpoint_df["gold_passage_ids"], checkpoint_df["rag_pids"])
print(f"RAG → MAP: {map_rag}, MRR: {mrr_rag}")


Baseline → MAP: 0.6844, MRR: 0.6844
RAG → MAP: 0.6628, MRR: 0.6628


**Recommendations:**

The baseline retriever performs slightly better than the RAG version in terms of both MAP and MRR, indicating that the passages it retrieves are more relevant (closer to the correct one) on average.


The baseline answers are slightly better than RAG on both overlap-based (ROUGE) and semantic (BERTScore) metrics.
The difference is small, but statistically meaningful depending on dataset size.

Query rewriting might be hurting retrieval precision.